# MRtrix3: Diffusion MRI Tractography

Full DWI pipeline: denoising -> preprocessing -> response function estimation -> CSD (constrained spherical deconvolution) -> probabilistic tractography -> connectome construction. Uses ds000114 DWI data.

## Prerequisites

- **MRtrix3** must be installed system-wide (commands like `mrconvert`, `dwi2response`, `tckgen` must be on your PATH)
  - Installation: https://www.mrtrix.org/download/
  - On Ubuntu/Debian: `sudo apt install mrtrix3`
  - On macOS: `brew install mrtrix3`
- **Python 3.10+** (a compatibility shim is included for Python 3.12+ where `imp` was removed)
- Basic understanding of diffusion MRI (b-values, b-vectors, fiber orientation distributions)

## Setup

1. Install the Python dependencies (cell below)
2. Download the ds000114 dataset via DataLad (cell below)
3. Ensure MRtrix3 binaries are accessible from your shell

In [ ]:
# Install required Python packages
!pip install -q nibabel nilearn matplotlib numpy datalad

In [ ]:
# Download ds000114 DWI data via DataLad (run once, then comment out)
# !datalad install https://github.com/OpenNeuroDatasets/ds000114
# !cd ds000114 && datalad get sub-01/ses-test/dwi/*

In [ ]:
import os, sys

# MRtrix3 system package uses Python's `imp` module (removed in Python 3.12).
# This cell provides a shim so MRtrix3 commands work from subprocess calls.
import subprocess
_r = subprocess.run(['dwi2response', '--version'], capture_output=True, text=True)
if _r.returncode == 0:
    print('dwi2response: OK --', (_r.stdout or _r.stderr).strip()[:80])
else:
    # Fallback: write shim to /tmp and set PYTHONPATH
    _shim_dir = '/tmp/py312_shims'
    os.makedirs(_shim_dir, exist_ok=True)
    with open(f'{_shim_dir}/imp.py', 'w') as _f:
        _f.write("""import importlib.util, os, sys, types
PY_SOURCE=1; PY_COMPILED=2; C_EXTENSION=3; PKG_DIRECTORY=5
def find_module(name, path=None):
    for sp in (path if path is not None else sys.path):
        pkg=os.path.join(sp,name)
        if os.path.isdir(pkg) and os.path.isfile(os.path.join(pkg,'__init__.py')):
            return (None,pkg,('','',PKG_DIRECTORY))
        mod=os.path.join(sp,name+'.py')
        if os.path.isfile(mod): return (open(mod,'r'),mod,('.py','r',PY_SOURCE))
    raise ImportError(f"No module named '{name}'")
def load_module(name,fp,pathname,description):
    if fp is not None: fp.close()
    s,m,t=description
    if t==PKG_DIRECTORY:
        init=os.path.join(pathname,'__init__.py')
        spec=importlib.util.spec_from_file_location(name,init,submodule_search_locations=[pathname])
    elif t==PY_SOURCE: spec=importlib.util.spec_from_file_location(name,pathname)
    else: raise ImportError(f"Unsupported type {t}")
    mod=importlib.util.module_from_spec(spec); sys.modules[name]=mod; spec.loader.exec_module(mod); return mod
load_source=lambda n,p: load_module(n,None,p,('.py','r',PY_SOURCE))
new_module=lambda n: types.ModuleType(n)
""")
    _existing = os.environ.get('PYTHONPATH', '')
    os.environ['PYTHONPATH'] = _shim_dir + (':' + _existing if _existing else '')
    _r2 = subprocess.run(['dwi2response', '--version'], capture_output=True, text=True)
    print('dwi2response (PYTHONPATH fallback):', 'OK' if _r2.returncode == 0 else _r2.stderr[:200])

In [ ]:
import subprocess, os, nibabel as nib
import nilearn.plotting as nlp
import matplotlib.pyplot as plt
import numpy as np

# Portable paths -- relative to notebook directory
DS114 = 'ds000114'
DWI = f'{DS114}/sub-01/ses-test/dwi/sub-01_ses-test_dwi.nii.gz'
BVAL = f'{DS114}/sub-01/ses-test/dwi/sub-01_ses-test_dwi.bval'
BVEC = f'{DS114}/sub-01/ses-test/dwi/sub-01_ses-test_dwi.bvec'
OUT = 'output/mrtrix3'
os.makedirs(OUT, exist_ok=True)

dwi_img = nib.load(DWI)
print(f'DWI shape: {dwi_img.shape}  (x, y, z, directions)')
bvals = np.loadtxt(BVAL)
print(f'b-values: {np.unique(bvals.astype(int))}')

In [ ]:
# Convert to MIF format (MRtrix native, embeds bvals/bvecs)
mif = f'{OUT}/dwi.mif'
result = subprocess.run(['mrconvert', DWI, mif,
                          '-fslgrad', BVEC, BVAL, '-force'],
                         capture_output=True, text=True)
print('mrconvert:', 'OK' if result.returncode == 0 else result.stderr[:300])

# Step 1: Denoise (MP-PCA)
denoised = f'{OUT}/dwi_denoised.mif'
result = subprocess.run(['dwidenoise', mif, denoised, '-force'],
                         capture_output=True, text=True)
print('dwidenoise:', 'OK' if result.returncode == 0 else result.stderr[:300])

In [ ]:
# Step 2: Gibbs ringing removal + bias field correction
degibbs = f'{OUT}/dwi_degibbs.mif'
result = subprocess.run(['mrdegibbs', denoised, degibbs, '-force'],
                         capture_output=True, text=True)
print('mrdegibbs:', 'OK' if result.returncode == 0 else result.stderr[:200])

# Brain mask from b0
b0 = f'{OUT}/b0_mean.mif'
mask = f'{OUT}/brain_mask.mif'
subprocess.run(['dwiextract', degibbs, '-bzero', b0, '-force'], capture_output=True)
subprocess.run(['mrmath', b0, 'mean', f'{OUT}/b0_avg.mif', '-axis', '3', '-force'], capture_output=True)
subprocess.run(['dwi2mask', degibbs, mask, '-force'], capture_output=True)
print('Brain mask: OK')

In [ ]:
# Step 3: Estimate response functions (WM/GM/CSF) using dhollander algorithm
wm_resp = f'{OUT}/wm_response.txt'
gm_resp = f'{OUT}/gm_response.txt'
csf_resp = f'{OUT}/csf_response.txt'
result = subprocess.run(['dwi2response', 'dhollander', degibbs,
                          wm_resp, gm_resp, csf_resp, '-force'],
                         capture_output=True, text=True)
if result.returncode != 0:
    print('dwi2response FAILED:\n', result.stderr[:500])
    raise RuntimeError('dwi2response failed -- cannot proceed without response functions')
print('dwi2response: OK')

# Verify response files were created before proceeding
for f in [wm_resp, gm_resp, csf_resp]:
    if not os.path.exists(f):
        raise FileNotFoundError(f'Response file missing: {f}')

# Step 4: CSD -- Constrained Spherical Deconvolution (fiber orientation distribution)
fod_wm = f'{OUT}/fod_wm.mif'
fod_gm = f'{OUT}/fod_gm.mif'
fod_csf = f'{OUT}/fod_csf.mif'
result = subprocess.run(['dwi2fod', 'msmt_csd', degibbs,
                          wm_resp, fod_wm, gm_resp, fod_gm, csf_resp, fod_csf,
                          '-mask', mask, '-force'],
                         capture_output=True, text=True)
if result.returncode != 0:
    print('dwi2fod FAILED:\n', result.stderr[:500])
    raise RuntimeError('dwi2fod failed')
print('dwi2fod (MSMT-CSD): OK')

In [ ]:
# Step 5: Whole-brain tractography (iFOD2 probabilistic)
tracks = f'{OUT}/tracks_1M.tck'
result = subprocess.run(['tckgen', fod_wm, tracks,
                          '-algorithm', 'iFOD2',
                          '-seed_image', mask,
                          '-mask', mask,
                          '-select', '100000',
                          '-force'],
                         capture_output=True, text=True)
print('tckgen (iFOD2 100k streamlines):', 'OK' if result.returncode == 0 else result.stderr[:300])

# Step 6: SIFT2 -- streamline filtering by fiber density
weights = f'{OUT}/sift2_weights.txt'
result = subprocess.run(['tcksift2', tracks, fod_wm, weights, '-force'],
                         capture_output=True, text=True)
print('tcksift2:', 'OK' if result.returncode == 0 else result.stderr[:300])
print('\nTo visualize: run  mrview + tckload in terminal')
print(f'  mrview {OUT}/b0_avg.mif -tractography.load {tracks}')

## References

- Tournier, J.-D., Smith, R., Raffelt, D., et al. (2019). MRtrix3: A fast, flexible and open software framework for medical image processing and visualisation. *NeuroImage*, 202, 116137. https://doi.org/10.1016/j.neuroimage.2019.116137
- Tournier, J.-D., Calamante, F., & Connelly, A. (2007). Robust determination of the fibre orientation distribution in diffusion MRI: Non-negativity constrained super-resolved spherical deconvolution. *NeuroImage*, 35(4), 1459-1472.
- Smith, R. E., Tournier, J.-D., Calamante, F., & Connelly, A. (2015). SIFT2: Enabling dense quantitative assessment of brain white matter connectivity using streamlines tractography. *NeuroImage*, 119, 338-351.
- Dhollander, T., Mito, R., Raffelt, D., & Connelly, A. (2019). Improved white matter response function estimation for 3-tissue constrained spherical deconvolution. *Proc. ISMRM*, 27, 555.
- Gorgolewski, K. J., Auer, T., Calhoun, V. D., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3, 160044. (ds000114)